# Day 2: Loss, Backpropagation and Training from Scratch (NumPy)

Aaj ka goal: loss functions samajhna, backprop likhna, MLP class banana (`forward`, `backward`, `update`), train karna, aur gradient check karna.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 1. Loss functions

- **MSE** (regression): `mean((y_hat - y)^2)`
- **Binary Cross-Entropy** (classification): `-mean(y*log(y_hat) + (1-y)*log(1-y_hat))`

Loss batata hai ke prediction kitni galat hai. Training ka kaam is number ko kam karna hai.

In [ ]:
def mse(y_hat, y):
    return np.mean((y_hat - y) ** 2)

def bce(y_hat, y, eps=1e-12):
    y_hat = np.clip(y_hat, eps, 1 - eps)   # log(0) se bachne ke liye
    return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

y_true = np.array([[1.0], [0.0], [1.0]])
good   = np.array([[0.9], [0.1], [0.8]])
bad    = np.array([[0.2], [0.9], [0.4]])

print("MSE  good:", mse(good, y_true).round(4), " bad:", mse(bad, y_true).round(4))
print("BCE  good:", bce(good, y_true).round(4), " bad:", bce(bad, y_true).round(4))

## 2. Gradient descent

`w = w - lr * dL/dw`

- `lr` (learning rate) bohat bada: loss uchhalta hai ya diverge karta hai.
- `lr` bohat chhota: training bohat slow.

## 3. Backpropagation

Chain rule se har weight ka gradient nikalte hain, output se input ki taraf peeche jate hue.

Network: `X -> Linear(W1,b1) -> ReLU -> Linear(W2,b2) -> Sigmoid -> BCE`

```
dZ2 = (A2 - y) / m          # sigmoid + BCE ka gradient simplify ho kar yehi banta hai
dW2 = A1.T @ dZ2
db2 = sum(dZ2)
dA1 = dZ2 @ W2.T
dZ1 = dA1 * (Z1 > 0)        # ReLU ka derivative
dW1 = X.T @ dZ1
db1 = sum(dZ1)
```

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

class MLP:
    def __init__(self, n_in, n_hidden, n_out, lr=0.1, seed=0):
        rng = np.random.default_rng(seed)
        self.lr = lr
        self.W1 = rng.standard_normal((n_in, n_hidden)) * np.sqrt(2 / n_in)
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.standard_normal((n_hidden, n_out)) * np.sqrt(2 / n_hidden)
        self.b2 = np.zeros((1, n_out))

    def forward(self, X):
        self.X = X
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = relu(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = sigmoid(self.Z2)
        return self.A2

    def loss(self, y_hat, y):
        return bce(y_hat, y)

    def backward(self, y):
        m = y.shape[0]
        dZ2 = (self.A2 - y) / m
        self.dW2 = self.A1.T @ dZ2
        self.db2 = dZ2.sum(axis=0, keepdims=True)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * (self.Z1 > 0)
        self.dW1 = self.X.T @ dZ1
        self.db1 = dZ1.sum(axis=0, keepdims=True)

    def update(self):
        self.W1 -= self.lr * self.dW1
        self.b1 -= self.lr * self.db1
        self.W2 -= self.lr * self.dW2
        self.b2 -= self.lr * self.db2

    def predict(self, X):
        return (self.forward(X) > 0.5).astype(int)

## 4. Train karo (make_moons)

In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
y = y.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = MLP(n_in=2, n_hidden=16, n_out=1, lr=0.5)

losses = []
for epoch in range(2000):
    y_hat = model.forward(X_train)
    losses.append(model.loss(y_hat, y_train))
    model.backward(y_train)
    model.update()
    if epoch % 400 == 0:
        print(f"epoch {epoch:4d}  loss {losses[-1]:.4f}")

train_acc = (model.predict(X_train) == y_train).mean()
test_acc  = (model.predict(X_test) == y_test).mean()
print(f"train acc: {train_acc:.3f}   test acc: {test_acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(losses)
plt.xlabel("epoch"); plt.ylabel("BCE loss"); plt.title("Training loss")
plt.grid(alpha=0.3)
plt.show()

def plot_boundary(model, X, y, title):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200),
                         np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.forward(grid).reshape(xx.shape)
    plt.figure(figsize=(6, 4))
    plt.contourf(xx, yy, zz, levels=20, cmap="RdBu", alpha=0.6)
    plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="RdBu", edgecolor="k", s=20)
    plt.title(title)
    plt.show()

plot_boundary(model, X_test, y_test, "Trained model (test data)")

## 5. Gradient check

Apna analytic gradient (backprop) ko numerical estimate se compare karte hain:

`(L(w + eps) - L(w - eps)) / (2 * eps)`

Relative error 1e-6 ya usse kam ho to backprop sahi hai.

In [ ]:
def gradient_check(model, X, y, param_name, n_checks=5, eps=1e-5):
    model.forward(X)
    model.backward(y)
    param = getattr(model, param_name)
    grad  = getattr(model, "d" + param_name)

    rng = np.random.default_rng(1)
    flat_idx = rng.choice(param.size, size=min(n_checks, param.size), replace=False)

    for fi in flat_idx:
        idx = np.unravel_index(fi, param.shape)
        old = param[idx]

        param[idx] = old + eps
        loss_plus = model.loss(model.forward(X), y)
        param[idx] = old - eps
        loss_minus = model.loss(model.forward(X), y)
        param[idx] = old

        numeric = (loss_plus - loss_minus) / (2 * eps)
        analytic = grad[idx]
        rel_err = abs(numeric - analytic) / max(1e-12, abs(numeric) + abs(analytic))
        print(f"{param_name}{tuple(int(i) for i in idx)}: analytic={analytic:+.8f}  numeric={numeric:+.8f}  rel_err={rel_err:.2e}")

check_model = MLP(2, 8, 1, seed=3)
Xc, yc = X_train[:50], y_train[:50]
for name in ["W1", "b1", "W2", "b2"]:
    gradient_check(check_model, Xc, yc, name, n_checks=3)

## 6. Experiments (khud karo)

1. `lr` ko 0.01, 0.1, 1, 5 karke loss curve compare karo.
2. `n_hidden` ko 2, 4, 16, 64 karo. Decision boundary kaise badalti hai?
3. Epochs 200 vs 5000, kya overfitting dikhti hai?
4. Backward me `(self.Z1 > 0)` hata do aur gradient check chalao. Check fail hona chahiye.

## 7. Notes (apne words me likho)

- Loss function kya hai aur MSE vs cross-entropy kab use karte hain:
- Gradient descent aur learning rate:
- Forward pass ek line me:
- Backward pass ek line me:
- Gradient check kyun karte hain: